# Training an LSTM-based Authorized Push Payment (APP) Fraud Detection Model
This notebook is used to prepare and preprocess the datasets from IBM Synthetic Datasets, which are then used for training an LSTM-based Authorized Push Payment (APP) fraud detection model. The datasets are loaded from four different CSV files and then combined into one DataFrame. This enriched dataset provides the features needed to train a fraud detection model that can learn patterns from account characteristics, bank metrics, and transaction behaviors. Such as:
- Transaction details (amount, currency, format, type)
- Fraud labels (Is_APP_Fraud)
- Sender account features (country, currency, entity type, overdraft, branch/bank metrics)
- Recipient account features (country, entity type)
The notebook then builds a preprocessing + LSTM model, trains it, and exports a self-contained ONNX file for inference.


In [ ]:
import os, io, sys
import pandas as pd
import numpy as np
import tensorflow as tf
import keras
from keras import layers, models, ops

from fraud_model_layers import (
    OnnxVocabOneHot, OnnxVocabOrdinal,
    CyclicalEncoding, LogTransform, TimeOfDayEncoding,
    focal_loss, TP, FP, FN, TN, F1Score,
    fix_onnx_export,
)

print(f"TensorFlow {tf.__version__}  |  Keras {keras.__version__}")

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["WRAPT_DISABLE_EXTENSIONS"] = "true"

datasets_dir   = "./datasets/"
model_save_dir = './saved_model'
os.makedirs(model_save_dir, exist_ok=True)

timesteps       = 7     # transactions per sequence
batch_size      = 2048
test_batch_size = 2000



## Data Preparation


This code loads and enriches bank transfer transaction data for fraud detection by joining three datasets: banks, accounts (people + companies), and transactions. It creates a comprehensive dataset where each transaction is enriched with contextual features from both the sender's account (account type, currency, branch size, bank statistics) and recipient's account (country, entity type), while filtering out cash transactions and converting data types for efficient processing. The final merged dataset contains transaction details plus account-level and bank-level features that help the LSTM model identify fraudulent Authorized Push Payment (APP) patterns.

In [ ]:
print("Loading datasets...")

banks_cols     = ["Bank_ID", "Num_Transactions", "Num_Total_Locations"]
accts_cols     = ["Financial_Institution_ID", "Financial_Institution_Name", "Branch",
                  "Account_ID", "Account_Country", "Account_Currency", "Entity_Type",
                  "Account_Type", "Max_Overdraft"]
bank_xfers_cols = ["Transaction_Number", "Transaction_Date", "Transaction_Time",
                   "Transaction_Day_of_Week", "From_Bank", "From_Account", "To_Bank",
                   "To_Account", "Amount_Paid", "Payment_Currency", "Payment_Format",
                   "Transaction_Type", "Is_Instant_Payments_Fraud", "Is_APP_Fraud",
                   "Sufficient_Funds", "Overdraft_Okay", "Is_Hold", "From_End_Balance"]

banks_df           = pd.read_csv(datasets_dir + "us_small_banks.csv",               usecols=banks_cols)
accts_people_df    = pd.read_csv(datasets_dir + "us_small_liquid_accts_people.csv",  usecols=accts_cols)
accts_companies_df = pd.read_csv(datasets_dir + "us_small_liquid_accts_companies.csv",
                                 encoding='iso8859_2', usecols=accts_cols)
bank_xfers_df      = pd.read_csv(datasets_dir + "us_small_bank_xfers-chrono.csv",   usecols=bank_xfers_cols)

accts_df = pd.concat([accts_people_df, accts_companies_df], ignore_index=True)
accts_df = accts_df.astype({"Account_Country": "category", "Account_Currency": "category",
                             "Entity_Type": "category", "Account_Type": "category", "Account_ID": str})

branches_df = (accts_df.groupby(["Financial_Institution_ID", "Branch"])["Account_ID"]
               .nunique().to_frame(name="Branch_Account_Count").reset_index())
branches_df = branches_df.merge(banks_df[["Bank_ID", "Num_Transactions", "Num_Total_Locations"]],
                                left_on="Financial_Institution_ID", right_on="Bank_ID", how="left")
branches_df.drop("Bank_ID", axis="columns", inplace=True)
branches_df.rename(columns={"Num_Transactions": "Bank_Num_Transactions",
                             "Num_Total_Locations": "Bank_Num_Total_Locations"}, inplace=True)
branches_df = branches_df.astype({"Bank_Num_Transactions": "Int64", "Bank_Num_Total_Locations": "Int64"})

accts_merged_df = accts_df.merge(branches_df, on=["Financial_Institution_ID", "Branch"], how="left")

bank_xfers_cleaned_df = bank_xfers_df[
    (bank_xfers_df["From_Bank"] != "Cash") & (bank_xfers_df["To_Bank"] != "Cash")
].astype({"From_Account": str, "To_Account": str, "Payment_Currency": "category",
           "Payment_Format": "category", "Transaction_Type": "category",
           "Is_APP_Fraud": bool, "Sufficient_Funds": bool, "Overdraft_Okay": bool, "Is_Hold": bool})
bank_xfers_cleaned_df.reset_index(drop=True, inplace=True)

from_cols = ["Financial_Institution_Name", "Account_ID", "Account_Country", "Account_Currency",
             "Entity_Type", "Account_Type", "Max_Overdraft",
             "Branch_Account_Count", "Bank_Num_Transactions", "Bank_Num_Total_Locations"]
bank_xfers_merged_df = bank_xfers_cleaned_df.merge(
    accts_merged_df[from_cols],
    left_on=["From_Bank", "From_Account"],
    right_on=["Financial_Institution_Name", "Account_ID"], how="left")
bank_xfers_merged_df.drop(["Financial_Institution_Name", "Account_ID"], axis="columns", inplace=True)
bank_xfers_merged_df.rename(columns={
    "Account_Country": "From_Account_Country", "Account_Currency": "From_Account_Currency",
    "Entity_Type": "From_Entity_Type", "Account_Type": "From_Account_Type",
    "Max_Overdraft": "From_Account_Max_Overdraft",
    "Branch_Account_Count": "From_Branch_Account_Count",
    "Bank_Num_Transactions": "From_Bank_Num_Transactions",
    "Bank_Num_Total_Locations": "From_Bank_Num_Total_Locations"}, inplace=True)

bank_xfers_merged_df["Timestamp"] = (bank_xfers_merged_df["Transaction_Date"].astype(str) + " " +
                                     bank_xfers_merged_df["Transaction_Time"].astype(str))

bank_xfers_merged_df = bank_xfers_merged_df.merge(
    accts_merged_df[["Financial_Institution_Name", "Account_ID", "Account_Country", "Entity_Type"]],
    left_on=["To_Bank", "To_Account"],
    right_on=["Financial_Institution_Name", "Account_ID"], how="left")
bank_xfers_merged_df.drop(["Financial_Institution_Name", "Account_ID"], axis="columns", inplace=True)
bank_xfers_merged_df.rename(columns={"Account_Country": "To_Account_Country",
                                     "Entity_Type": "To_Entity_Type"}, inplace=True)
bank_xfers_merged_df.reset_index(drop=True, inplace=True)

dataset_df = bank_xfers_merged_df
print(f"Dataset shape: {dataset_df.shape}  |  Fraud cases: {dataset_df['Is_APP_Fraud'].sum()}")



## Feature Engineering


In [ ]:
print("Applying transformations...")

dataset_df["Timestamp"]   = pd.to_datetime(dataset_df["Timestamp"], format="%Y-%m-%d %H:%M:%S.%f")
dataset_df["Month"]       = dataset_df["Timestamp"].dt.month
dataset_df["Day_Of_Month"]= dataset_df["Timestamp"].dt.day
dataset_df["Day_Of_Week"] = dataset_df["Timestamp"].dt.dayofweek
dataset_df["Hour"]        = dataset_df["Timestamp"].dt.hour
dataset_df["Minute"]      = dataset_df["Timestamp"].dt.minute
dataset_df["Is_Weekday"]  = dataset_df["Timestamp"].dt.dayofweek.isin([0,1,2,3,4]).astype(int)

for col in ["From_Account_Max_Overdraft", "From_Branch_Account_Count",
            "From_Bank_Num_Transactions", "From_Bank_Num_Total_Locations"]:
    dataset_df[col] = dataset_df[col].fillna(0)

for col in ["Sufficient_Funds", "Overdraft_Okay", "Is_Hold"]:
    dataset_df[col] = dataset_df[col].astype(int)

# Column groups used by build_preprocessing_model, prepare_data_dict, and input_spec.
# Boolean columns are cast to int above so all float_cols share a single dtype here.
float_cols = [
    'Month', 'Day_Of_Month', 'Day_Of_Week', 'Hour', 'Minute', 'Is_Weekday',
    'Sufficient_Funds', 'Overdraft_Okay', 'Is_Hold',
    'Amount_Paid', 'From_End_Balance', 'From_Account_Max_Overdraft',
    'From_Branch_Account_Count', 'From_Bank_Num_Transactions', 'From_Bank_Num_Total_Locations',
]
categorical_cols = ["From_Entity_Type", "From_Account_Type", "To_Entity_Type",
                    "Payment_Format", "Transaction_Type", "From_Account_Country",
                    "From_Account_Currency", "To_Account_Country", "Payment_Currency"]
for col in categorical_cols:
    if isinstance(dataset_df[col].dtype, pd.CategoricalDtype):
        if "Unknown" not in dataset_df[col].cat.categories:
            dataset_df[col] = dataset_df[col].cat.add_categories(["Unknown"])
    dataset_df[col] = dataset_df[col].fillna("Unknown").astype(str)

print("Feature engineering complete!")

### Custom Keras Layers for Preprocessing
The custom preprocessing layers (`CyclicalEncoding`, `LogTransform`, `TimeOfDayEncoding`) encode temporal patterns for fraud detection while remaining ONNX-exportable.
`OnnxVocabOneHot` and `OnnxVocabOrdinal` are ONNX-compatible replacements for `StringLookup` — they use only `tf.equal` / `tf.cast` (opset-13 primitives).
All five layers are defined in `fraud_model_layers.py` and imported in the first cell above.

## Build a Keras model that includes all preprocessing layers


In [ ]:
def build_preprocessing_model():
    """Functional Keras model: 24 raw inputs → normalised feature vector."""
    inputs = {col: layers.Input(shape=(1,), dtype=tf.float32, name=col) for col in float_cols}
    inputs.update({col: layers.Input(shape=(1,), dtype=tf.string,  name=col) for col in categorical_cols})
    processed = []

    # Cyclical temporal features
    cyclical = layers.Concatenate(name='cyclical_concat')([
        CyclicalEncoding(12, name='month_cyclical')(inputs['Month']),
        CyclicalEncoding(31, name='day_cyclical')(inputs['Day_Of_Month']),
        CyclicalEncoding(7,  name='dow_cyclical')(inputs['Day_Of_Week']),
        TimeOfDayEncoding(name='time_of_day_cyclical')(
            layers.Concatenate(name='time_concat')([inputs['Hour'], inputs['Minute']])),
    ])
    processed.append(layers.Normalization(name='cyclical_norm')(cyclical))

    # Boolean pass-through
    processed.append(layers.Concatenate(name='boolean_concat')([
        inputs['Is_Weekday'], inputs['Sufficient_Funds'],
        inputs['Overdraft_Okay'], inputs['Is_Hold'],
    ]))

    # Log-transformed amounts
    amounts = layers.Concatenate(name='amounts_concat')([
        LogTransform(name='amount_paid_log')(inputs['Amount_Paid']),
        LogTransform(name='balance_log')(inputs['From_End_Balance']),
        LogTransform(name='overdraft_log')(inputs['From_Account_Max_Overdraft']),
    ])
    processed.append(layers.Normalization(name='amounts_norm')(amounts))

    # Count features
    counts = layers.Concatenate(name='counts_concat')([
        inputs['From_Branch_Account_Count'],
        inputs['From_Bank_Num_Transactions'],
        inputs['From_Bank_Num_Total_Locations'],
    ])
    processed.append(layers.Normalization(name='counts_norm')(counts))

    # Low-cardinality → one-hot
    for feat in ['From_Entity_Type', 'From_Account_Type', 'To_Entity_Type',
                 'Payment_Format', 'Transaction_Type']:
        processed.append(OnnxVocabOneHot(name=f'{feat}_lookup')(inputs[feat]))

    # High-cardinality → ordinal + normalise
    for feat in ['From_Account_Country', 'From_Account_Currency',
                 'To_Account_Country', 'Payment_Currency']:
        processed.append(
            layers.Normalization(name=f'{feat}_norm')(
                OnnxVocabOrdinal(name=f'{feat}_lookup')(inputs[feat])))

    out = layers.Concatenate(name='feature_concat')(processed)
    return models.Model(inputs=inputs, outputs=out, name='preprocessing'), inputs


preprocessing_model, feature_inputs = build_preprocessing_model()
print("Preprocessing model built.")
preprocessing_model.summary()

### Adapt preprocessing layers with actual data
Adapting preprocessing layers with actual data is essential because these layers need to learn statistics from your training data to work correctly.

In [ ]:
print("Adapting preprocessing layers...")

def prepare_data_dict(df):
    """DataFrame → dict of TF tensors keyed by feature name."""
    d = {col: tf.constant(df[col].values.reshape(-1, 1), dtype=tf.float32)
         for col in float_cols}
    for col in categorical_cols:
        d[col] = tf.constant(df[col].values.reshape(-1, 1), dtype=tf.string)
    return d

sample_data = prepare_data_dict(dataset_df.iloc[:min(10000, len(dataset_df))])

vocab_map = {feat: sample_data[feat] for feat in categorical_cols}
for layer in preprocessing_model.layers:
    if isinstance(layer, (OnnxVocabOneHot, OnnxVocabOrdinal)):
        for feat, data in vocab_map.items():
            if feat in layer.name:
                layer.adapt(data)
                print(f"  Adapted {layer.name}")
                break

cyclical_sample = layers.Concatenate()([
    CyclicalEncoding(12)(sample_data['Month']),
    CyclicalEncoding(31)(sample_data['Day_Of_Month']),
    CyclicalEncoding(7)(sample_data['Day_Of_Week']),
    TimeOfDayEncoding()(layers.Concatenate()([sample_data['Hour'], sample_data['Minute']]))
])
preprocessing_model.get_layer('cyclical_norm').adapt(cyclical_sample)

amounts_sample = layers.Concatenate()([
    LogTransform()(sample_data['Amount_Paid']),
    LogTransform()(sample_data['From_End_Balance']),
    LogTransform()(sample_data['From_Account_Max_Overdraft'])
])
preprocessing_model.get_layer('amounts_norm').adapt(amounts_sample)

counts_sample = layers.Concatenate()([
    sample_data['From_Branch_Account_Count'],
    sample_data['From_Bank_Num_Transactions'],
    sample_data['From_Bank_Num_Total_Locations']
])
preprocessing_model.get_layer('counts_norm').adapt(counts_sample)

print("  Adapted cyclical_norm, amounts_norm, counts_norm")

test_output = preprocessing_model(sample_data)
preprocessed_feature_size = test_output.shape[-1]
print(f"Normalization adapted. Preprocessed feature size: {preprocessed_feature_size}")

## Build LSTM Model
This code builds a complete fraud detection model by creating sequence inputs for 7 timesteps of transaction history, individually applying the preprocessing model to each timestep (since TimeDistributed doesn't support dictionary inputs), and concatenating the results into a properly shaped tensor for the LSTM layers. The architecture consists of two stacked LSTM layers (200 units each) that process the preprocessed sequences to capture temporal patterns, followed by a sigmoid output layer that predicts fraud probability for each timestep. The entire model (preprocessing + LSTM) is unified into a single Keras model that can be exported to ONNX for deployment.

In [ ]:
# Sequence inputs: (batch, timesteps, feature_dim)
sequence_inputs = {
    key: layers.Input(shape=(timesteps,) + feature_inputs[key].shape[1:],
                      dtype=feature_inputs[key].dtype, name=key)
    for key in feature_inputs
}

# Apply preprocessing per timestep, then concatenate
preprocessed_timesteps = []
for t in range(timesteps):
    t_inputs = {k: ops.squeeze(sequence_inputs[k][:, t:t+1, :], axis=1) for k in sequence_inputs}
    preprocessed_timesteps.append(ops.expand_dims(preprocessing_model(t_inputs), axis=1))

preprocessed_sequences = layers.Reshape(
    (timesteps, preprocessed_feature_size), name='reshape_sequences')(
    layers.Concatenate(axis=1, name='concat_timesteps')(preprocessed_timesteps))

lstm_out = layers.LSTM(200, return_sequences=True, name='lstm_1')(preprocessed_sequences)
lstm_out = layers.LSTM(200, return_sequences=True, name='lstm_2')(lstm_out)
output   = layers.Dense(1, activation='sigmoid', name='output')(lstm_out)

complete_model = models.Model(inputs=sequence_inputs, outputs=output, name='fraud_detection_lstm')
print("Complete model built.")
complete_model.summary()



## Prepare Training Data


In [ ]:
print("Preparing training data...")

labels = dataset_df['Is_APP_Fraud'].values.astype(np.float32)
total  = len(labels)

# ── Chronological split ───────────────────────────────────────────────────────
# Reverted to chronological split: the stratified split caused score-distribution
# collapse (fraud and genuine both scoring ~0.068) because interspersed fraud
# indices forced sequence windows to cross split boundaries, giving the model
# incoherent context and causing it to converge to the base-rate constant.
# Chronological order is correct for time-series: the model trains on the past
# and is validated on more recent transactions.
train_size    = int(total * 0.5)
val_size      = int(total * 0.3)
train_indices = np.arange(train_size)
val_indices   = np.arange(train_size, train_size + val_size)
test_indices  = np.arange(train_size + val_size, total)

print(f"Split — Train: {len(train_indices)}  Val: {len(val_indices)}  Test: {len(test_indices)}")

train_label_slice = labels[train_indices]
genuine_count    = int((train_label_slice == 0).sum())
fraud_count      = int((train_label_slice == 1).sum())
raw_class_weight = float(genuine_count / fraud_count)

# Cap the class weight — the raw genuine/fraud ratio is often 200–500×, which
# pushes the decision boundary so aggressively that precision collapses
# (val_FP >> val_TN). A cap of 50 retains strong recall pressure while keeping
# false positives at a manageable level.
CLASS_WEIGHT_CAP      = 50.0
positive_class_weight = min(raw_class_weight, CLASS_WEIGHT_CAP)
print(f"Training class distribution — genuine: {genuine_count}  fraud: {fraud_count}")
print(f"Raw class weight: {raw_class_weight:.1f}  →  capped at: {positive_class_weight:.1f}")

def create_sequences_dict(df, indices, labels, timesteps):
    """Build windowed sequence arrays from a DataFrame."""
    valid = indices[indices >= timesteps - 1]
    if len(valid) == 0:
        return None, None
    full = prepare_data_dict(df)
    seq_data = {}
    for key, tensor in full.items():
        feat = tensor.numpy()
        seqs = (np.empty  if feat.dtype == np.object_ else np.zeros)(
            (len(valid), timesteps) + feat.shape[1:],
            dtype=feat.dtype if feat.dtype != np.object_ else object)
        for i, idx in enumerate(valid):
            for t in range(timesteps):
                seqs[i, t] = feat[idx - (timesteps - 1 - t)]
        seq_data[key] = (tf.constant(seqs, dtype=tf.string)
                         if seqs.dtype == object else seqs)
    seq_labels = np.zeros((len(valid), timesteps, 1), dtype=np.float32)
    for i, idx in enumerate(valid):
        for t in range(timesteps):
            seq_labels[i, t, 0] = labels[idx - (timesteps - 1 - t)]
    return seq_data, seq_labels


print("Creating sequences...")
train_seq_data, train_seq_labels = create_sequences_dict(dataset_df, train_indices, labels, timesteps)
val_seq_data,   val_seq_labels   = create_sequences_dict(dataset_df, val_indices,   labels, timesteps)
test_seq_data,  test_seq_labels  = create_sequences_dict(dataset_df, test_indices,  labels, timesteps)
print(f"Train {train_seq_labels.shape}  Val {val_seq_labels.shape}  Test {test_seq_labels.shape}")



## Compile and Train
`TP`/`FP`/`FN`/`TN`, `F1Score`, and `focal_loss` are imported from `fraud_model_layers.py`. All confusion-matrix metrics use `EVAL_THRESHOLD=0.2` (also from the module) because focal loss compresses fraud scores well below 0.5. The best model checkpoint is selected by training F1 (rather than `val_F1`) because the validation period covers a later calendar window with a distinct fraud-pattern distribution.

In [ ]:
complete_model.compile(
    optimizer='adam',
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy', TP(name='TP'), FP(name='FP'), FN(name='FN'), TN(name='TN'),
             keras.metrics.TruePositives(name='tp'), keras.metrics.FalsePositives(name='fp'),
             keras.metrics.FalseNegatives(name='fn'), keras.metrics.TrueNegatives(name='tn'),
             F1Score(name='F1')]
)

checkpoint_dir = "./checkpoints/app_fraud_lstm_keras_preprocessing/"
os.makedirs(checkpoint_dir, exist_ok=True)

# Saves a checkpoint every epoch for audit/comparison
cp_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_dir + "iter-{epoch:02d}.weights.h5",
    save_weights_only=True, verbose=1)

# ── train_F1, not val_F1 as the validation set covers a later chronological window whose fraud patterns
# differ from the training window (temporal distribution shift) and val_F1 stays at 0.0.
best_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_dir + "best.weights.h5",
    monitor='F1', mode='max',
    save_best_only=True, save_weights_only=True, verbose=1)

early_stop = keras.callbacks.EarlyStopping(
    monitor='F1', mode='max',
    patience=3,                # stop after 3 epochs of no train_F1 improvement
    restore_best_weights=True, # restore the best training-F1 epoch on exit
    verbose=1)

print("Training model...")

# With focal loss the alpha parameter handles class weighting internally.
# sample_weight is kept uniform (all 1.0) to avoid double-penalising fraud
# and re-introducing the base-rate attractor.
train_sample_weights = np.ones_like(train_seq_labels, dtype=np.float32)
val_sample_weights   = np.ones_like(val_seq_labels,   dtype=np.float32)

history = complete_model.fit(
    train_seq_data, train_seq_labels,
    batch_size=batch_size, epochs=50,  # set high; EarlyStopping will intervene
    validation_data=(val_seq_data, val_seq_labels, val_sample_weights),
    callbacks=[cp_callback, best_callback, early_stop], verbose=1,
    sample_weight=train_sample_weights)



## Save and Evaluate


In [ ]:
print(f"Saving model to {model_save_dir}...")
complete_model.save(os.path.join(model_save_dir, "fraud_detection_model.keras"))
complete_model.save_weights(os.path.join(model_save_dir, "model_weights.weights.h5"))

print("Evaluating on test set...")
test_results = complete_model.evaluate(test_seq_data, test_seq_labels,
                                        batch_size=test_batch_size, verbose=1)
for name, value in zip(complete_model.metrics_names, test_results):
    print(f"  {name}: {value:.4f}")

## Export to ONNX
`tf2onnx.convert.from_keras` was removed in TF 2.16 / Keras 3. We wrap the model in a `tf.Module` so `from_function` can trace through a single dict input.


In [ ]:
print("Exporting unified model to ONNX...")

import tf2onnx

onnx_output_path = os.path.join(model_save_dir, "fraud_detection_unified.onnx")

# All Keras input names are PascalCase/Title_Snake; serving names are the same in lowercase.
input_spec = {inp.name.lower(): tf.TensorSpec(shape=[None] + list(inp.shape[1:]),
                                               dtype=inp.dtype, name=inp.name.lower())
              for inp in complete_model.inputs}

class _ServingWrapper(tf.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    @tf.function(input_signature=[input_spec])
    def __call__(self, inputs):
        keras_inputs = {inp.name: inputs[inp.name.lower()] for inp in self.model.inputs}
        return self.model(keras_inputs, training=False)

serving_wrapper = _ServingWrapper(complete_model)
model_proto, _ = tf2onnx.convert.from_function(
    serving_wrapper.__call__,
    input_signature=[input_spec], opset=13, output_path=onnx_output_path)

print("Post-processing ONNX graph: fixing tf2onnx export bugs...")
fix_onnx_export(model_proto, preprocessing_model, onnx_output_path)

print(f"✓ Unified model exported to {onnx_output_path}")
print("  Includes: preprocessing, LSTM layers, output layer")